# Channel Visualization with Textures

In [ ]:
import pickle, MeshFEM, mesh, wall_generation, visualization, numpy as np, parametrization
(sdfVertices, sdfTris, sdf) = pickle.load(open('stripe_sdf_ns8_f130.pkl', 'rb'))

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.01,
                                              minContourLen=0.075)

In [ ]:
import visualization, importlib
importlib.reload(visualization)
tex = visualization.line_segments_texture(pts, edges, width=12.4, dpi=72)

In [ ]:
lilium = mesh.Mesh('../examples/lilium.msh')
uv = np.loadtxt('data/lilium_tower_parametrization_5_1_2019.txt')

import registration
uv3D = np.pad(uv, [(0, 0), (0, 1)], 'constant')
R, t = registration.register_points(lilium.vertices(), uv3D)
uv3D = uv3D @ R.transpose() + t

In [ ]:
import tri_mesh_viewer
importlib.reload(tri_mesh_viewer)
from tri_mesh_viewer import TriMeshViewer, TextureMap, FlatteningAnimation

texMap = TextureMap(uv, tex, normalizeUV=True, powerOfTwo=True)
fa = FlatteningAnimation(lilium, uv3D, width=1024, height=768, textureMap=texMap)
fa.show()

In [ ]:
fa.viewer.materialLibrary.materials['solid_vcFalse_tex377af90b074f4c3391a087e2fc86401e'].color = 'white'

In [ ]:
m, fuseMarkers = wall_generation.triangulate_channel_walls(pts[:,0:2], edges, 0.0001)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=20, height=18)

In [ ]:
import visualization, importlib
importlib.reload(visualization)
visualization.plot_line_segments(pts, edges, width=20, height=16)

In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet.visualizationMesh(), width=1024, height=768)
viewer.showWireframe()
viewer.show()

In [ ]:
isheet.setUseTensionFieldEnergy(False)

In [ ]:
mkdir lilium_inflate

In [ ]:
import time
isheet.pressure = 40
inflation.benchmark_reset()
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)
    if cr.numIters() < iterations_per_output: break
    isheet.writeDebugMesh('lilium_inflate/inflation_p40_step_{}.msh'.format(step))
    #viewer.update(False, isheet.visualizationMesh())
    time.sleep(0.05) # Allow some mesh synchronization time for pythreejs
inflation.benchmark_report()

In [ ]:
isheet.tensionStateHistogram()